# **10 ВАРИАНТ - InceptionV3** Танюшкин А.Л.

# Лабораторная работа №3. Классификация изображений CNN

	Цель лабораторной работы – Классификация изображений с использованием свёрточных нейронных сетей.

1.Определить какую сеть вам нужно будет загрузить согласно вашему номеру варианта:

# InceptionV3

2.Загрузить сеть, предобученную на ImageNet (https://keras.io/api/applications/)

In [1]:
import os
import numpy as np
import pandas as pd

import io
import zipfile
import logging

import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.applications import InceptionV3 # type: ignore
from tensorflow.keras.applications.inception_v3 import preprocess_input, decode_predictions # type: ignore
from tensorflow.keras.preprocessing import image # type: ignore

from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Activation, Dense # type: ignore
from tensorflow.keras.preprocessing.image import ImageDataGenerator # type: ignore

from tensorflow.keras.layers import GlobalAveragePooling2D # type: ignore

In [63]:
model = InceptionV3(include_top = True,
                    weights='imagenet',
                    input_shape = (299, 299, 3),
                    classes = 1000
                    )

3.Выполнить классификацию изображения на загруженной сети, найденного в интернете. Изображение должно показывать один объект одного из следующих классов: лошадь, собака, кошка, самолёт, корабль, автомобиль, дом

In [64]:
def classify_img():
    for file in os.listdir('task2/'):
        file_p = os.path.join('task2/', file)

        img = image.load_img(file_p, target_size=(299, 299))

        img_arr = image.img_to_array(img)
        img_arr = np.expand_dims(img_arr, axis=0)
        img_arr = preprocess_input(img_arr)

        pred = model.predict(img_arr)
        decod_pred = decode_predictions(pred, top=3)[0]

        for i, prediction in enumerate(decod_pred):
            print(f"{i+1}. {prediction[1]}: {prediction[2]:.2%}            Файл: {file}")
        

In [65]:
classify_img()

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1. minivan: 33.12%            Файл: images.jpeg
2. sports_car: 4.97%            Файл: images.jpeg
3. pole: 4.28%            Файл: images.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step
1. washer: 96.44%            Файл: images12.jpeg
2. ashcan: 0.15%            Файл: images12.jpeg
3. stove: 0.13%            Файл: images12.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
1. moving_van: 10.55%            Файл: images2.jpeg
2. minibus: 4.73%            Файл: images2.jpeg
3. ambulance: 3.53%            Файл: images2.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 223ms/step
1. bullet_train: 26.99%            Файл: images3.jpeg
2. passenger_car: 24.86%            Файл: images3.jpeg
3. streetcar: 15.77%            Файл: images3.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1. minivan: 13.17%            Файл: images4.jpeg
2. moving_van: 5.04%            Файл: images4.jpeg
3. limousine: 4.59%            Файл: images4.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1. cab: 20.78%         

4.	Загрузить два набора данных horses_or_humans, cats_vs_dogs

In [2]:
(train_cat_dog, test_cat_dog) = tfds.load('cats_vs_dogs',
                                split=['train[:70%]', 'train[70%:]'],
                                shuffle_files=True, 
                                as_supervised=True)

In [3]:
(train_horse_human, test_horse_human) = tfds.load('horses_or_humans',
                                split=['train', 'test'],
                                shuffle_files=True,
                                as_supervised=True)

In [4]:
def predobr(image, label):
    image = tf.image.resize(image, [224, 224])
    image = tf.cast(image, tf.float32) / 255.0

    return image, label

In [5]:
train_cat_dog = train_cat_dog.map(predobr).batch(32).prefetch(tf.data.AUTOTUNE)
test_cat_dog = test_cat_dog.map(predobr).batch(32).prefetch(tf.data.AUTOTUNE)

In [6]:
train_horse_human = train_horse_human.map(predobr).batch(32).prefetch(tf.data.AUTOTUNE)
test_horse_human = test_horse_human.map(predobr).batch(32).prefetch(tf.data.AUTOTUNE)

In [71]:
train_cat_dog

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>

In [72]:
train_horse_human

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>

7.Создать собственную сеть (полностью своя сеть со своими слоями и архитектурой), обучить данную сеть с помощью генераторов на двух наборах данных: horses_or_humans, cats_vs_dogs. Оценить точность классификации сети. (за качеством не гонимся) 

In [73]:
model = Sequential()

model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(MaxPooling2D(2, 2))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Dropout(0.25))

model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Dropout(0.25))

model.add(Conv2D(256, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Dropout(0.25))

model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

c:\Users\Aleggg\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [74]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
    )

In [75]:
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_573 (Conv2D)             │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_34 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_574 (Conv2D)             │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_35 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_575 (Conv2D)             │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_36 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_22 (Dropout)            │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_576 (Conv2D)             │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_37 (MaxPooling2D) │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_23 (Dropout)            │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 512)            │    18,874,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_24 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,263,809 (73.49 MB)

 Trainable params: 19,263,809 (73.49 MB)

 Non-trainable params: 0 (0.00 B)

In [76]:
history_horses = model.fit(
    train_horse_human,
    epochs=10,
    validation_data=test_horse_human,
    verbose=1
)

Epoch 1/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 35s 992ms/step - accuracy: 0.5083 - loss: 0.9649 - val_accuracy: 0.5000 - val_loss: 0.6926
Epoch 2/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.6855 - loss: 0.6094 - val_accuracy: 0.5000 - val_loss: 0.7054
Epoch 3/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 33s 981ms/step - accuracy: 0.7819 - loss: 0.4583 - val_accuracy: 0.5273 - val_loss: 0.7152
Epoch 4/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 32s 980ms/step - accuracy: 0.8802 - loss: 0.2907 - val_accuracy: 0.5977 - val_loss: 0.8840
Epoch 5/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 33s 982ms/step - accuracy: 0.9231 - loss: 0.2021 - val_accuracy: 0.8203 - val_loss: 0.5810
Epoch 6/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 33s 994ms/step - accuracy: 0.9679 - loss: 0.0900 - val_accuracy: 0.6719 - val_loss: 2.1301
Epoch 7/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 32s 971ms/step - accuracy: 0.9883 - loss: 0.0389 - val_accuracy: 0.6055 - val_loss: 3.3194
Epoch 8/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 33s 984ms/step - accuracy: 0.9864 - loss: 0.0527 - val_accurac

In [77]:
test_loss_horses, test_acc_horses = model.evaluate(test_horse_human)
print(f'точность horses_or_humans: {test_acc_horses:.2%}')

8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 199ms/step - accuracy: 0.7852 - loss: 1.8467
точность horses_or_humans: 78.52%


In [78]:
model2 = Sequential()

model2.add(Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model2.add(MaxPooling2D(2, 2))
model2.add(Dropout(0.25))

model2.add(Conv2D(64, (3, 3), activation='relu'))
model2.add(MaxPooling2D(2, 2))
model2.add(Dropout(0.25))

model2.add(Conv2D(128, (3, 3), activation='relu'))
model2.add(MaxPooling2D(2, 2))
model2.add(Dropout(0.25))

model2.add(Conv2D(256, (3, 3), activation='relu'))
model2.add(MaxPooling2D(2, 2))
model2.add(Dropout(0.25))

model2.add(Flatten())
model2.add(Dense(512, activation='relu'))
model2.add(Dropout(0.5))
model2.add(Dense(1, activation='sigmoid'))

In [79]:
model2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
    )

In [80]:
model2.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_577 (Conv2D)             │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_38 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_25 (Dropout)            │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_578 (Conv2D)             │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_39 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_26 (Dropout)            │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_579 (Conv2D)             │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_40 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_27 (Dropout)            │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_580 (Conv2D)             │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_41 (MaxPooling2D) │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_28 (Dropout)            │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 512)            │    18,874,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_29 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,263,809 (73.49 MB)

 Trainable params: 19,263,809 (73.49 MB)

 Non-trainable params: 0 (0.00 B)

In [81]:
history_cat = model2.fit(
    train_cat_dog,
    epochs=5,
    validation_data=test_cat_dog,
    verbose=1
)

Epoch 1/5
509/509 ━━━━━━━━━━━━━━━━━━━━ 536s 1s/step - accuracy: 0.5864 - loss: 0.6653 - val_accuracy: 0.6402 - val_loss: 0.6314
Epoch 2/5
509/509 ━━━━━━━━━━━━━━━━━━━━ 533s 1s/step - accuracy: 0.7170 - loss: 0.5555 - val_accuracy: 0.7339 - val_loss: 0.5274
Epoch 3/5
509/509 ━━━━━━━━━━━━━━━━━━━━ 534s 1s/step - accuracy: 0.7732 - loss: 0.4761 - val_accuracy: 0.7919 - val_loss: 0.4409
Epoch 4/5
509/509 ━━━━━━━━━━━━━━━━━━━━ 533s 1s/step - accuracy: 0.8083 - loss: 0.4261 - val_accuracy: 0.8109 - val_loss: 0.4122
Epoch 5/5
509/509 ━━━━━━━━━━━━━━━━━━━━ 538s 1s/step - accuracy: 0.8258 - loss: 0.3903 - val_accuracy: 0.8162 - val_loss: 0.3995


In [82]:
test_loss_cats, test_acc_cats = model2.evaluate(test_cat_dog)
print(f'точность cats_vs_dogs: {test_acc_cats:.2%}')

219/219 ━━━━━━━━━━━━━━━━━━━━ 45s 203ms/step - accuracy: 0.8162 - loss: 0.3995
точность cats_vs_dogs: 81.62%


8.Загрузить сеть, предобученную на InceptionV3 по варианту из пункта 1, выполнить дообучение этой сети на двух наборах данных: horses_or_humans, cats_vs_dogs. Оценить точность классификации новой (составной) сети. (А вот здесь с качеством можно поработать)

In [7]:
model_inc = InceptionV3(
    weights='imagenet', 
    include_top=False, 
    input_shape=(299, 299,3)
)
model_inc.trainable = False

In [103]:
def create_models():
    model3 = Sequential()

    model3.add(model_inc)
    model3.add(tf.keras.layers.GlobalAveragePooling2D())
    model3.add(Dense(512, activation='relu'))
    model3.add(Dropout(0.3))
    model3.add(Dense(256, activation='relu'))
    model3.add(Dropout(0.3))
    model3.add(Dense(1, activation='sigmoid'))

    return model3

In [94]:
def preprocess_for_inc(image, label):
    image = tf.image.resize(image, [299, 299])
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    
    return image, label

In [ ]:
create_model = create_models()

In [96]:
train_horse_inc = train_horse_human.map(preprocess_for_inc).prefetch(tf.data.AUTOTUNE)
test_horse_inc = test_horse_human.map(preprocess_for_inc).prefetch(tf.data.AUTOTUNE)

In [97]:
create_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [107]:
create_model.summary()

Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inception_v3 (Functional)       │ (None, 8, 8, 2048)     │    21,802,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_6      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_32 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_33 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,344,805 (96.68 MB)

 Trainable params: 1,180,673 (4.50 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

 Optimizer params: 2,361,348 (9.01 MB)

In [98]:
history_horse_inc = create_model.fit(
    train_horse_inc,
    epochs=10,
    validation_data=test_horse_inc,
    verbose=1
)

Epoch 1/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 89s 2s/step - accuracy: 0.6037 - loss: 0.6664 - val_accuracy: 0.7188 - val_loss: 0.5277
Epoch 2/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - accuracy: 0.7760 - loss: 0.4622 - val_accuracy: 0.8281 - val_loss: 0.4401
Epoch 3/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 79s 2s/step - accuracy: 0.8442 - loss: 0.3601 - val_accuracy: 0.7930 - val_loss: 0.5820
Epoch 4/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 82s 2s/step - accuracy: 0.8929 - loss: 0.2533 - val_accuracy: 0.7930 - val_loss: 0.6826
Epoch 5/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 83s 2s/step - accuracy: 0.9056 - loss: 0.2242 - val_accuracy: 0.7812 - val_loss: 0.7542
Epoch 6/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 82s 2s/step - accuracy: 0.9036 - loss: 0.2106 - val_accuracy: 0.7578 - val_loss: 0.9127
Epoch 7/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 83s 2s/step - accuracy: 0.9309 - loss: 0.1755 - val_accuracy: 0.7344 - val_loss: 1.1894
Epoch 8/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 69s 2s/step - accuracy: 0.9289 - loss: 0.1682 - val_accuracy: 0.6953 - val_loss:

In [100]:
test_loss_hum, test_acc_hum = create_model.evaluate(test_horse_inc)
print(f'точность horses_or_humans: {test_acc_hum:.2%}')

8/8 ━━━━━━━━━━━━━━━━━━━━ 12s 2s/step - accuracy: 0.7422 - loss: 1.2369
точность horses_or_humans: 74.22%


In [104]:
create_model2 = create_models()

In [105]:
train_cat_dog_inc = train_cat_dog.map(preprocess_for_inc).prefetch(tf.data.AUTOTUNE)
test_cat_dog_inc = test_cat_dog.map(preprocess_for_inc).prefetch(tf.data.AUTOTUNE)

In [106]:
create_model2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [108]:
history_cat_dog_inc = create_model2.fit(
    train_cat_dog_inc,
    epochs=5,
    validation_data=test_cat_dog_inc,
    verbose=1
)

Epoch 1/5
509/509 ━━━━━━━━━━━━━━━━━━━━ 1256s 2s/step - accuracy: 0.6168 - loss: 0.6526 - val_accuracy: 0.6644 - val_loss: 0.6094
Epoch 2/5
509/509 ━━━━━━━━━━━━━━━━━━━━ 1253s 2s/step - accuracy: 0.6703 - loss: 0.6054 - val_accuracy: 0.6981 - val_loss: 0.5802
Epoch 3/5
509/509 ━━━━━━━━━━━━━━━━━━━━ 1335s 3s/step - accuracy: 0.6827 - loss: 0.5961 - val_accuracy: 0.7172 - val_loss: 0.5849
Epoch 4/5
509/509 ━━━━━━━━━━━━━━━━━━━━ 1448s 3s/step - accuracy: 0.6858 - loss: 0.5902 - val_accuracy: 0.7124 - val_loss: 0.5606
Epoch 5/5
509/509 ━━━━━━━━━━━━━━━━━━━━ 1261s 2s/step - accuracy: 0.6956 - loss: 0.5782 - val_accuracy: 0.7077 - val_loss: 0.5658


In [109]:
test_loss_cat, test_acc_cat = create_model2.evaluate(train_cat_dog_inc)
print(f'точность cat_or_dog: {test_acc_cat:.2%}')

509/509 ━━━━━━━━━━━━━━━━━━━━ 854s 2s/step - accuracy: 0.7150 - loss: 0.5605
точность cat_or_dog: 71.50%


9.Оценка классификации должна соответствовать следующим требованиям:

a.	Должна быть confusion matrix

b.	Должны быть показаны метрики accuracy, recall, precision, f1

c.	Должна быть построена PR кривая для каждого класса

# Доработанный вариант в файле "lab3(3_0).ipynb"